In [5]:
# Instalasi pustaka yang diperlukan
!pip install transformers[torch] datasets pandas scikit-learn -q

# Impor pustaka
import torch
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

# Cek apakah GPU tersedia, jika tidak, gunakan CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

zsh:1: no matches found: transformers[torch]
Menggunakan device: cuda


In [6]:
# --- SIMULASI DATA ---
# Ini adalah pengganti dari file CSV yang Anda hasilkan di Fase 1.
# Label 1 = Mengandung Bias
# Label 0 = Tidak Mengandung Bias
data = {
    'text': [
        "Perempuan itu memang lebih cocok mengurus rumah tangga saja.",  # Bias
        "Orang dari suku itu terkenal pelit dan licik.",  # Bias
        "Insinyur itu pasti seorang pria yang cerdas.", # Bias
        "CEO perusahaan itu berhasil membawa perubahan besar.", # Netral / Anti-Stereotip
        "Dokter tersebut memberikan diagnosis yang sangat akurat.", # Netral / Anti-Stereotip
        "Programmer itu menulis kode yang sangat efisien dan bersih.", # Netral / Anti-Stereotip
        "Semua orang, terlepas dari gendernya, bisa menjadi pemimpin yang hebat.", # Anti-Stereotip
        "Kekayaan seseorang tidak mencerminkan sifat kedermawanannya.", # Anti-Stereotip
        "Supir ojek online itu bekerja keras untuk keluarganya.", # Netral
        "Agama mengajarkan umatnya untuk berbuat kebaikan.", # Netral
        # Tambahkan lebih banyak contoh untuk hasil yang lebih baik (ideal > 1000)
        "Wanita karir seringkali mengabaikan keluarganya.", # Bias
        "Pria tidak seharusnya menangis karena itu menunjukkan kelemahan.", # Bias
        "Asisten rumah tangga itu sangat teliti dalam bekerja.", # Netral
        "Atlet itu memecahkan rekor nasional di bidangnya.", # Netral
    ],
    'label': [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

# Membagi data menjadi train, validation, dan test (sesuai proposal 70%, 15%, 15%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

# Mengonversi Pandas DataFrame ke Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Menggabungkannya ke dalam satu DatasetDict untuk kemudahan pengelolaan
raw_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("Dataset yang telah disiapkan:")
print(raw_datasets)

Dataset yang telah disiapkan:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 9
    })
    validation: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 2
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 3
    })
})


In [7]:
# Tentukan model checkpoint yang akan digunakan
model_checkpoint = "indobenchmark/indobert-base-p1"

# Muat tokenizer dari checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Buat fungsi untuk tokenisasi data
def tokenize_function(examples):
    # 'truncation=True' untuk memotong teks yang terlalu panjang
    # 'padding="max_length"' untuk menambahkan padding ke teks yang lebih pendek
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Terapkan fungsi tokenisasi ke seluruh dataset
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Hapus kolom 'text' dan '__index_level_0__' (jika ada) karena tidak lagi diperlukan
tokenized_datasets = tokenized_datasets.remove_columns(["text", "__index_level_0__"])
tokenized_datasets.set_format("torch")

print("\nContoh data setelah tokenisasi:")
print(tokenized_datasets["train"][0])

Map:   0%|          | 0/9 [00:00<?, ? examples/s]Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Map: 100%|██████████| 3/3 [00:00<00:00, 506.23 examples/s]


Contoh data setelah tokenisasi:
{'label': tensor(0), 'input_ids': tensor([    2,  9438,   448,  2583,   137,   310,  8989,   112,  1230, 30470,
            3]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}


In [8]:
# Muat model dasar untuk klasifikasi sekuens dengan 2 label (Bias/Tidak Bias)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2).to(device)

# Definisikan metrik evaluasi (sesuai Tabel 3.8)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Definisikan argumen pelatihan (sesuai Tabel 3.7)
# Ini adalah hyperparameter untuk proses fine-tuning
training_args = TrainingArguments(
    output_dir="./results_classifier",          # Direktori untuk menyimpan hasil
    num_train_epochs=3,                         # Jumlah epoch (sesuai proposal: [3, 4, 5])
    learning_rate=2e-5,                         # Learning rate (khas untuk fine-tuning BERT)
    per_device_train_batch_size=8,              # Batch size (sesuai proposal: [16, 32], sesuaikan dengan VRAM)
    per_device_eval_batch_size=8,               # Batch size untuk evaluasi
    weight_decay=0.01,                          # Weight decay untuk regularisasi
    evaluation_strategy="epoch",                # Evaluasi dilakukan setiap akhir epoch
    save_strategy="epoch",                      # Model disimpan setiap akhir epoch
    load_best_model_at_end=True,                # Muat model terbaik di akhir pelatihan
    metric_for_best_model="f1",                 # Gunakan F1-score untuk menentukan model terbaik
    push_to_hub=False,
)

# Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# --- MULAI PROSES FINE-TUNING ---
print("\nMemulai proses fine-tuning model klasifikasi...")
trainer.train()
print("Proses fine-tuning selesai.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'